In [ ]:
# @title Environment setup
%set_env MLFLOW_TRACKING_URI https://dagshub.com/m09/landscape-classifier.mlflow
%set_env MPLBACKEND notebook
!rm -rf sample_data .config
!pip install -U uv
!git config --global user.email "jane@doe.eu"
!git config --global user.name "Jane Doe"
!git config --global init.defaultBranch main
!git clone --branch main-en https://github.com/shuuchuu/landscape-classifier.git
%cd /content/landscape-classifier
!uv pip install --system -e ".[fastapi]"

# MLflow, model registry & deployment

## MLflow credentials

Enter your DagsHub credentials (the password is the secret token, not your DagsHub password):

In [ ]:
username = ""
token = ""

In [ ]:
# @title To run
%set_env MLFLOW_TRACKING_USERNAME {username}
%set_env MLFLOW_TRACKING_PASSWORD {token}

## Data retrieval

In [ ]:
!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s landscape
!mv landscape/seg_train train-data

## Publishing a model to a model registry

Modify the `src/landscape_classifier/train.py` file to train a model and track metrics with [MLflow Tracking](https://mlflow.org/docs/latest/tracking.html). [Keras](https://mlflow.org/docs/latest/python_api/mlflow.keras.html) pages on the MLflow Python API & pages on the [Keras flavor](https://mlflow.org/docs/latest/models.html#keras-keras) of MLflow models will be useful.

Then publish the trained model to the model registry at the end of the run, as `dev.ml.landscape-classifier`. [This page](https://mlflow.org/docs/latest/models.html#keras-keras) covers it, as well as [this one](https://mlflow.org/docs/latest/model-registry.html#adding-an-mlflow-model-to-the-model-registry).

The following cell runs the training:

In [ ]:
!landscape-classifier

## Using the MLflow model registry

Using the MLflow API, perform the following tasks on the model registry.

### Creating an MLflow client

Create an MLflow client and display the tracking address to check that it is correctly read from the environment variable defined at the top of this notebook.

In [ ]:
# Your code here

#### Solution

In [ ]:
import mlflow

MODEL_NAME = "dev.ml.landscape-classifier"

client = mlflow.MlflowClient()
client.tracking_uri

### Model search

Display all the registered models and, for each of them, its most recently created version.

In [ ]:
# Your code here

#### Solution

In [ ]:
def latest_version(name: str) -> str:
  versions = client.search_model_versions(f"name = '{name}'")
  return str(max(int(v.version) for v in versions))


for registered_model in client.search_registered_models():
  print(registered_model.name, latest_version(registered_model.name))

### Retrieving a model from the model registry

Now that you know how to get the name and version of the registered models from Python, retrieve one and check that it is the object you expect.

A model with several *flavors* can be loaded in several ways. What difference does it make?

In [ ]:
# Your code here

#### Solution

In [ ]:
from pathlib import Path

version = latest_version(MODEL_NAME)
model = mlflow.pyfunc.load_model(model_uri=f"models:/{MODEL_NAME}/{version}")
print(list(model.metadata.flavors))
model.predict([Path("forest1.jpg").read_bytes(), Path("forest2.jpg").read_bytes()])

The solution's model is published as a custom `pyfunc` model: its only flavor is `python_function`, which gives the whole pipeline (image preprocessing, prediction, result formatting). A model published with the Keras flavor can also be loaded as a Keras model (`mlflow.keras.load_model`), which only gives the network, without the preprocessing.

### Adding an alias & tags

Aliases and tags are a way to organize models in an MLflow model registry. Aliases always target a specific version of a model while tags can also apply to all versions of a model.

Add a `champion` alias to the last version of your model, a `domain` tag with value `cv` to your model, and a `size` tag with value `small` to the last version of your model.

In [ ]:
# Your code here

Now retrieve your model by using its alias rather than its exact version.

In [ ]:
# Your code here

#### Solution

In [ ]:
client.set_registered_model_alias(MODEL_NAME, "champion", version)
client.set_registered_model_tag(MODEL_NAME, "domain", "cv")
client.set_model_version_tag(MODEL_NAME, version, "size", "small")

In [ ]:
model = mlflow.pyfunc.load_model(model_uri=f"models:/{MODEL_NAME}@champion")
model.predict([Path("forest1.jpg").read_bytes()])

### Changing model phases

When you want to promote a model in your production pipeline, the currently recommended way is to copy the model to a new model name.

Copy the `champion` version of your model to a new name, where the `dev.` prefix is replaced by `staging.`.

In [ ]:
# Your code here

#### Solution

In [ ]:
client.copy_model_version(
  src_model_uri=f"models:/{MODEL_NAME}@champion",
  dst_name="staging.ml.landscape-classifier",
)

## API creation with FastAPI

Code all the questions of this part in the `src/landscape_classifier/api.py` file.

### Creating a preprocessing function suited to inference

Create a function that loads an image in the same format as the one used during training. What is the point that needs extreme care at this stage?

### Creating a model loading function

Create a function that retrieves the previously trained model from the MLflow model registry. This function takes the URI of the model to retrieve as input.

### Creating a FastAPI result class

FastAPI uses the excellent [Pydantic](https://docs.pydantic.dev/latest/) library to handle the input and output types of HTTP requests.

Create a class modeling the return type with the Pydantic library. The class should contain at least the predicted class (the one with the highest probability), and a dictionary of the classes and their probability according to the model.

### Creating the FastAPI API

Implement the inference API with a POST method that accepts an image file. [This documentation page](https://fastapi.tiangolo.com/tutorial/request-files/) can help.

## Dockerfile creation

Write a Dockerfile that:

- uses the Python 3.12 base image;
- installs `uv` as shown in class;
- installs the project's dependencies;
- copies the project's files;
- sets up an entry point that starts the FastAPI server.

Docker isn't available in Colab: on a machine with Docker, we build then run the image, passing it the MLflow credentials as environment variables:

    docker build -t shuuchuu/landscape-classifier .
    docker run --rm -it -e MLFLOW_TRACKING_URI -e MLFLOW_TRACKING_USERNAME \
      -e MLFLOW_TRACKING_PASSWORD -p 8000:80 shuuchuu/landscape-classifier

The API documentation, which also serves as a test client, is then at <http://localhost:8000/docs>.

## API improvements

How could the API be modified to be better suited to production deployment with a Docker image?

*Your answer here.*

## Solution

For all the questions without a solution, see [the `solution-en` branch of the GitHub repository](https://github.com/shuuchuu/landscape-classifier/tree/solution-en) of this lab.